# LangChain 提示词工程

本文基于 `langchain==1.3.0`，记录 LangChain 中提示词工程与 `system prompt` 的基础用法。

主要内容：
- 什么是提示词工程，它为什么会出现
- `system prompt` 通常如何组织
- Few-Shot Example适合解决什么问题
- 什么场景需要结构化输出，以及如何实现

## 一、提示词工程的简介与由来

提示词工程（Prompt Engineering）指的是：围绕具体任务，设计和组织传给模型的输入内容，使模型的输出更可控、更稳定。

这个概念的出现，本质上是因为大语言模型虽然能力很强，但对输入表达方式非常敏感。同一个任务，不同的提示方式，往往会直接影响输出质量。


In [ ]:
# 加载环境变量，并初始化后续示例会复用的模型
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

load_dotenv()

# 这里统一走百炼兼容端点，初始化一个基础聊天模型
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    base_url=os.environ["DASHSCOPE_BASE_URL"],
    api_key=os.environ["DASHSCOPE_API_KEY"],
    temperature=0,
)


In [ ]:
# 反面例子：只给一个很短的角色描述
bad_agent = create_agent(
    model=model,
    system_prompt="你是一名物业服务助手，回答问题时请保持简洁。",
)

# 直接提问，观察模型在缺少背景和规则时的自然回复
result = bad_agent.invoke({
    "messages": [{"role": "user", "content": "我家门禁卡丢了，租户能不能代办补卡，需要带什么？"}]
})

print(result["messages"][-1].content)


上面的回答偏离真实业务太远，更稳的做法是把特定小区的特定要求作为基准，
让智能体有角色、有任务、有规则/约束、有背景信息、有输出要求的去回答用户。

## 二、System Prompt 的结构化设计
下面继续用同一个物业助手，对比结构化 `system prompt` 在“范围内回答”和“超出范围婉拒”这两类问题上的表现。


In [ ]:
# 正面例子：把角色、规则、背景和输出要求组合到一个 system prompt 中
system_prompt = """
# 角色
你是一名给人亲切感的，专业的物业服务助手。

# 任务
负责回答停车、门禁、报修、缴费等问题。

# 规则 / 约束
1. 如果问题属于物业服务范围，就直接回答。
2. 如果问题不属于物业服务范围，要明确婉拒。
3. 不要回答与物业无关的话题。

# 背景信息
- 门禁卡补办需要登记业主姓名、房号和联系电话。
- 如为租户代办，需要租户身份证，还需要业主授权信息。
- 补办完成时间通常以物业前台通知为准。

# 输出要求
1. 先直接回答用户问题。
2. 再列出需要准备的材料。
3. 回复要简洁、清楚。
"""

# 用同一个 Agent 处理物业范围内的问题
good_agent = create_agent(model=model, system_prompt=system_prompt)
result = good_agent.invoke({
    "messages": [
        {"role": "user", "content": "我家门禁卡丢了，租户能不能代办补卡，需要带什么？"}
    ]
})
print(result["messages"][-1].content)

print("\n" + "=" * 50 + "\n")

# 再用同一个 Agent 处理超出物业范围的问题
result = good_agent.invoke({
    "messages": [
        {"role": "user", "content": "给我写一篇800字的高考作文。"}
    ]
})
print(result["messages"][-1].content)


这两个例子的差异，不在于哪个回复的“字数更多”本身，而在于：后者基于事实依据把任务与要求组织得更具体、更准确。

一句话，`system prompt`没有固定的定义，它是一个持续优化的过程，而一个好的 `system prompt`，通常会把下面几类信息说清楚：
- 角色
- 任务
- 规则 / 约束
- 背景信息
- 输出要求
- 示例（可选）

## 三、Few-Shot 示例的适用场景与效果

Few-Shot 更适合这种任务：模型不是“会不会回答”，而是“能不能按你指定的风格和形式稳定输出”。

下面用一个更直观的场景：给模型一段文字，让它先自由表达理解；再通过 Few-Shot，让它模仿苏轼的口吻写一首短诗。


In [ ]:
# 准备一段景物描写，先看模型自由发挥时会怎么理解
user_text = "昨夜江边起了很大的风，吹得芦苇东倒西歪。天快亮时，风停了，江面忽然平静下来，只剩一轮淡月还挂在天边。"

# 不使用 Few-Shot：模型自由表达感受和理解
agent_no_few_shot = create_agent(
    model=model,
    system_prompt="""
你是一名写作助手。

请阅读用户给出的文字，并写出你的感受与理解。控制在300字以内。
""",
)

result = agent_no_few_shot.invoke({
    "messages": [{"role": "user", "content": user_text}]
})

print(result["messages"][-1].content)


In [ ]:
# 同一段文字，改用 Few-Shot 约束输出风格、篇幅和形式
user_text = "昨夜江边起了很大的风，吹得芦苇东倒西歪。天快亮时，风停了，江面忽然平静下来，只剩一轮淡月还挂在天边。"

# 使用 Few-Shot：要求模仿苏轼口吻，输出一首四句七言短诗
agent_few_shot = create_agent(
    model=model,
    system_prompt="""
你是一名古典诗词写作助手。

请阅读用户给出的文字，模仿苏轼的口吻写一首七言绝句。
要求：
1. 只输出诗句，不要解释。
2. 一共4句。
3. 每句7个字。

示例：
user: 夜雨初歇，远山被云雾半遮，江上有一只小船缓缓飘过。
assistant: 雨洗空山翠欲流
雾深江阔一舟浮
人间多少风波事
都付烟波自在秋

user: 秋天傍晚，落日照着长桥，桥下流水很慢，几片黄叶顺流而去。
assistant: 残照长桥水自闲
几枚黄叶下前湾
平生不为悲秋住
且向斜晖看远山
""",
)

result = agent_few_shot.invoke({
    "messages": [{"role": "user", "content": user_text}]
})

print(result["messages"][-1].content)


## 四、结构化输出的实现方式

结构化输出适合这样的场景：**模型返回的结果，不只是给人看，还要给程序继续处理。**

为了更直观地看出区别，先看同一个问题在“不使用结构化输出”时的自然回复，再看结构化输出后的结果。

实现方式很直接：先定义一个继承 `BaseModel` 的结构化类，再在 `create_agent(..., response_format=你的类)` 中声明输出结构，调用后从 `result["structured_response"]` 中获取结果。

这里单独使用一个关闭 thinking 的模型，是因为百炼部分模型在 thinking mode 下不适合直接配合 `response_format`。


In [ ]:
# 先看不使用结构化输出时，模型会怎样自然回复
agent_plain_reply = create_agent(
    model=model,
    system_prompt="你是一名物业工单分析助手。",
)

# 这里返回的是自然语言，适合直接展示给用户
result = agent_plain_reply.invoke({
    "messages": [{"role": "user", "content": "我家住 2 号楼 1 单元 602，联系方式是 13812345678。家里卫生间一直漏水，今天下午 19:00 方便，请安排师傅上门来维修。"}]
})

print(result["messages"][-1].content)


自然回复更适合直接给人看，但如果后面还要做分类、入库、页面渲染或 API 返回，仅靠自然语言就不够稳定。

这时更适合把结果限制到固定字段中。


In [ ]:
from pydantic import BaseModel, Field

# 单独初始化一个关闭 thinking 的模型，用来演示结构化输出
non_thinking_model = init_chat_model(
    model="qwen3.5-122b-a10b",
    model_provider="openai",
    base_url=os.environ["DASHSCOPE_BASE_URL"],
    api_key=os.environ["DASHSCOPE_API_KEY"],
    temperature=0,
    extra_body={"enable_thinking": False},
)

# 用 BaseModel 定义结构化输出字段
class PropertyTicketAnalysis(BaseModel):
    """物业工单分析结果。"""
    category: str = Field(description="工单类别，如停车、门禁、报修、缴费")
    urgency: str = Field(description="紧急程度，如高、中、低")
    address: str = Field(description="报修地址")
    contact_phone: str = Field(description="联系电话")
    available_time: str = Field(description="用户方便上门的时间")
    issue_description: str = Field(description="故障描述")
    suggested_reply: str = Field(description="建议回复")


In [ ]:
# 把结构化类传给 response_format，限制模型按固定字段返回结果
agent_structured = create_agent(
    model=non_thinking_model,
    system_prompt="你是一名物业工单分析助手，请把用户输入中的关键信息提取到固定字段中。",
    response_format=PropertyTicketAnalysis,
)

# 调用后，结构化结果会出现在 structured_response 中
result = agent_structured.invoke({
    "messages": [{"role": "user", "content": "我家住 2 号楼 1 单元 602，联系方式是 13812345678。家里卫生间一直漏水，今天下午 19:00 方便，请安排师傅上门来维修。"}]
})

# 以 JSON 形式打印，方便观察字段结构
print(result["structured_response"].model_dump_json(indent=2))


### 本篇小结

- 提示词工程的核心，是把任务要求组织成稳定、可控的模型输入。
- 一个更稳的 `system prompt`，通常会把角色、任务、规则 / 约束、背景信息、输出要求说明清楚。
- Few-Shot 更适合控制风格、格式和输出形式。
- 当结果还要继续给程序处理时，优先考虑结构化输出。